# Analyse Exploratoire des Données - Stations Piézométriques

Cette analyse examine les 18 stations piézométriques pour comprendre:
- La stationnarité des séries temporelles
- Les patterns de saisonnalité
- Les corrélations entre variables
- Les relations de causalité
- Les délais (lags) optimaux pour la prédiction

**Variables disponibles:**
- `date`: Date de la mesure
- `level`: Niveau piézométrique (cible)
- `PRELIQ_Q`: Précipitation liquide
- `T_Q`: Température
- `ETP_Q`: Évapotranspiration

## 1. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Statistical tests
from scipy import stats
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf, grangercausalitytests
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.seasonal import STL

# Darts (only for TimeSeries and check_seasonality)
from darts import TimeSeries
from darts.utils.statistics import check_seasonality

# IPython display
from IPython.display import display, Markdown, HTML

# Visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

# Create output directory for figures
figs_dir = Path('e:/junon-time-series/figs')
figs_dir.mkdir(exist_ok=True)

display(Markdown("✅ **Imports completed**"))

## 2. Load ALL 18 Stations

In [ ]:
# Load all piezometric stations (skip missing files)
data_dir = Path('e:/junon-time-series/data/piezos')
stations = {}
station_info = []

for i in range(1, 19):
    file_path = data_dir / f'piezo{i}.csv'
    
    # Skip if file doesn't exist
    if not file_path.exists():
        continue
    
    df = pd.read_csv(file_path, parse_dates=['date'])
    df = df.sort_values('date').reset_index(drop=True)
    
    stations[f'piezo{i}'] = df
    
    # Collect info
    station_info.append({
        'Station': f'piezo{i}',
        'Samples': len(df),
        'Start Date': df['date'].min(),
        'End Date': df['date'].max(),
        'Duration (days)': (df['date'].max() - df['date'].min()).days,
        'Missing Values': df.isnull().sum().sum()
    })

# Create summary table
summary_df = pd.DataFrame(station_info)

# Display styled summary
display(Markdown(f"### Summary of {len(stations)} Piezometric Stations"))
display(Markdown(f"**Total samples:** {summary_df['Samples'].sum():,} | **Date range:** {summary_df['Start Date'].min().date()} to {summary_df['End Date'].max().date()}"))

# Style the dataframe
styled_summary = summary_df.style\
    .format({
        'Samples': '{:,}',
        'Duration (days)': '{:,}',
        'Missing Values': '{:,}'
    })\
    .background_gradient(subset=['Samples'], cmap='Blues')\
    .background_gradient(subset=['Duration (days)'], cmap='Greens')\
    .applymap(lambda x: 'background-color: #ffcccc' if x > 0 else '', subset=['Missing Values'])

display(styled_summary)

## 3. Focus Analysis on Piezo1 (Representative)

Nous effectuons une analyse détaillée sur piezo1 comme exemple représentatif, puis nous comparerons les résultats à travers toutes les stations.

In [ ]:
# Focus on piezo1 for detailed analysis
df_piezo1 = stations['piezo1'].copy()

# Remove any missing values
df_piezo1 = df_piezo1.dropna()

# Create info DataFrame
piezo1_info = pd.DataFrame([
    {'Property': 'Shape', 'Value': f'{df_piezo1.shape[0]} rows × {df_piezo1.shape[1]} columns'},
    {'Property': 'Date Range', 'Value': f'{df_piezo1["date"].min().date()} to {df_piezo1["date"].max().date()}'},
    {'Property': 'Duration', 'Value': f'{(df_piezo1["date"].max() - df_piezo1["date"].min()).days:,} days'},
    {'Property': 'Frequency', 'Value': 'Daily'},
    {'Property': 'Missing Values', 'Value': '0 (cleaned)'}
])

display(Markdown("### Piezo1 Dataset Information"))
display(piezo1_info.style.hide(axis='index').set_properties(**{'text-align': 'left'}))

# Display statistics with styling
display(Markdown("### Basic Statistics"))
stats_df = df_piezo1[['level', 'PRELIQ_Q', 'T_Q', 'ETP_Q']].describe().T
stats_df.columns = ['Count', 'Mean', 'Std', 'Min', '25%', '50%', '75%', 'Max']

styled_stats = stats_df.style\
    .format('{:.3f}')\
    .background_gradient(subset=['Mean'], cmap='coolwarm')\
    .background_gradient(subset=['Std'], cmap='YlOrRd')

display(styled_stats)

In [ ]:
# Visualize all variables over time
fig, axes = plt.subplots(4, 1, figsize=(14, 12))

variables = [
    ('level', 'Niveau Piézométrique (m)', 'blue'),
    ('PRELIQ_Q', 'Précipitation (mm)', 'green'),
    ('T_Q', 'Température (°C)', 'red'),
    ('ETP_Q', 'Évapotranspiration (mm)', 'orange')
]

for ax, (var, label, color) in zip(axes, variables):
    ax.plot(df_piezo1['date'], df_piezo1[var], color=color, linewidth=0.8, alpha=0.7)
    ax.set_ylabel(label, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_xlim(df_piezo1['date'].min(), df_piezo1['date'].max())

axes[-1].set_xlabel('Date', fontweight='bold')
axes[0].set_title('Piezo1: Évolution Temporelle des Variables', fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig(figs_dir / 'eda_piezo1_timeseries.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Stationarity Tests (Piezo1)

Tests de stationnarité pour évaluer si les séries temporelles ont des propriétés statistiques constantes dans le temps.

**Tests utilisés:**
- **ADF (Augmented Dickey-Fuller)**: H₀ = non-stationary (reject if p < 0.05)
- **KPSS (Kwiatkowski-Phillips-Schmidt-Shin)**: H₀ = stationary (reject if p < 0.05)

In [ ]:
def test_stationarity(series, name):
    """
    Perform ADF and KPSS tests for stationarity
    
    ADF: H0 = non-stationary (reject if p < 0.05)
    KPSS: H0 = stationary (reject if p < 0.05)
    """
    # Remove NaN values
    series = series.dropna()
    
    # ADF test
    adf_result = adfuller(series, autolag='AIC')
    adf_statistic = adf_result[0]
    adf_pvalue = adf_result[1]
    
    # KPSS test
    kpss_result = kpss(series, regression='ct', nlags='auto')
    kpss_statistic = kpss_result[0]
    kpss_pvalue = kpss_result[1]
    
    return {
        'Variable': name,
        'ADF Stat': adf_statistic,
        'ADF p-val': adf_pvalue,
        'ADF Result': 'Stationary' if adf_pvalue < 0.05 else 'Non-Stationary',
        'KPSS Stat': kpss_statistic,
        'KPSS p-val': kpss_pvalue,
        'KPSS Result': 'Stationary' if kpss_pvalue >= 0.05 else 'Non-Stationary'
    }

# Test all variables
stationarity_results = []
for var in ['level', 'PRELIQ_Q', 'T_Q', 'ETP_Q']:
    result = test_stationarity(df_piezo1[var], var)
    stationarity_results.append(result)

# Create DataFrame
stationarity_df = pd.DataFrame(stationarity_results)

# Display with styling
display(Markdown("### Stationarity Test Results"))

def color_pvalue(val):
    """Color p-values: green if < 0.05, red otherwise"""
    if isinstance(val, (int, float)):
        return 'background-color: #90EE90' if val < 0.05 else 'background-color: #FFB6C6'
    return ''

def color_result(val):
    """Color result: green if stationary, red if non-stationary"""
    if val == 'Stationary':
        return 'background-color: #90EE90; font-weight: bold'
    elif val == 'Non-Stationary':
        return 'background-color: #FFB6C6; font-weight: bold'
    return ''

styled_stationarity = stationarity_df.style\
    .format({
        'ADF Stat': '{:.4f}',
        'ADF p-val': '{:.4f}',
        'KPSS Stat': '{:.4f}',
        'KPSS p-val': '{:.4f}'
    })\
    .applymap(color_pvalue, subset=['ADF p-val', 'KPSS p-val'])\
    .applymap(color_result, subset=['ADF Result', 'KPSS Result'])

display(styled_stationarity)

# Create visualization
fig, ax = plt.subplots(figsize=(10, 5))

variables = stationarity_df['Variable'].tolist()
x = np.arange(len(variables))
width = 0.35

# Count stationary results
adf_stationary = [1 if r == 'Stationary' else 0 for r in stationarity_df['ADF Result']]
kpss_stationary = [1 if r == 'Stationary' else 0 for r in stationarity_df['KPSS Result']]

bars1 = ax.bar(x - width/2, adf_stationary, width, label='ADF', color='#4CAF50', alpha=0.8)
bars2 = ax.bar(x + width/2, kpss_stationary, width, label='KPSS', color='#2196F3', alpha=0.8)

ax.set_xlabel('Variable', fontweight='bold')
ax.set_ylabel('Stationary (1) / Non-Stationary (0)', fontweight='bold')
ax.set_title('Stationarity Test Summary', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(variables)
ax.set_ylim(0, 1.2)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(figs_dir / 'eda_piezo1_stationarity.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Seasonality Detection (Piezo1)

Détection de saisonnalité à différentes périodes (hebdomadaire, mensuelle, annuelle).

In [ ]:
# Convert to Darts TimeSeries for seasonality check
ts_level = TimeSeries.from_dataframe(
    df_piezo1[['date', 'level']].set_index('date'),
    value_cols='level'
)

# Check seasonality at different periods
periods_to_test = [
    (7, 'Hebdomadaire'),
    (30, 'Mensuelle'),
    (365, 'Annuelle')
]

seasonality_results = []
for period, name in periods_to_test:
    # Set max_lag to at least 2*period to avoid errors
    max_lag = max(period * 2, 400)
    is_seasonal = check_seasonality(ts_level, m=period, max_lag=max_lag, alpha=0.05)
    
    # Get test statistic for display
    from statsmodels.tsa.stattools import acf
    acf_values = acf(ts_level.values().flatten(), nlags=min(period+10, len(ts_level)-1))
    test_stat = acf_values[period] if period < len(acf_values) else 0
    
    seasonality_results.append({
        'Period': f'{period} days',
        'Type': name,
        'Detected': 'Yes' if is_seasonal else 'No',
        'ACF at Period': test_stat
    })

# Create DataFrame
seasonality_df = pd.DataFrame(seasonality_results)

# Display with styling
display(Markdown("### Seasonality Detection Results"))

def color_detection(val):
    if val == 'Yes':
        return 'background-color: #90EE90; font-weight: bold'
    elif val == 'No':
        return 'background-color: #FFE4B5'
    return ''

styled_seasonality = seasonality_df.style\
    .format({'ACF at Period': '{:.4f}'})\
    .applymap(color_detection, subset=['Detected'])

display(styled_seasonality)

# Visualization
fig, ax = plt.subplots(figsize=(8, 5))

detected = [1 if d == 'Yes' else 0 for d in seasonality_df['Detected']]
colors = ['#4CAF50' if d == 1 else '#FFA726' for d in detected]

bars = ax.bar(seasonality_df['Type'], detected, color=colors, alpha=0.7, edgecolor='black')
ax.set_ylabel('Seasonal (1) / Not Seasonal (0)', fontweight='bold')
ax.set_title('Seasonality Detection by Period', fontsize=14, fontweight='bold')
ax.set_ylim(0, 1.2)
ax.grid(True, alpha=0.3, axis='y')

# Add labels on bars
for bar, period in zip(bars, seasonality_df['Period']):
    height = bar.get_height()
    label = 'Seasonal' if height > 0.5 else 'Not Seasonal'
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.05,
            f'{label}\n({period})',
            ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(figs_dir / 'eda_piezo1_seasonality.png', dpi=300, bbox_inches='tight')
plt.show()

# Note
if not any(r['Detected'] == 'Yes' for r in seasonality_results):
    display(Markdown("**Note:** Pas de saisonnalité détectée - c'est normal si la série est dominée par la tendance"))

## 6. STL Decomposition (Piezo1)

Décomposition STL (Seasonal-Trend-Loess) pour séparer tendance, saisonnalité et résidus.

**Note**: Nous utilisons `statsmodels.tsa.seasonal.STL` directement (pas `extract_trend_and_seasonality()` de Darts).

In [ ]:
# Prepare data for STL
df_stl = df_piezo1.set_index('date')['level'].copy()

# STL Decomposition using statsmodels
stl = STL(df_stl, seasonal=365, trend=730)
result = stl.fit()

# Extract components
trend = result.trend
seasonal = result.seasonal
residual = result.resid

# Calculate variance contribution
var_original = df_stl.var()
var_trend = trend.var()
var_seasonal = seasonal.var()
var_residual = residual.var()
var_total = var_trend + var_seasonal + var_residual

# Create variance contribution DataFrame
variance_df = pd.DataFrame([
    {'Component': 'Trend', 'Variance': var_trend, 'Contribution (%)': 100*var_trend/var_total},
    {'Component': 'Seasonal', 'Variance': var_seasonal, 'Contribution (%)': 100*var_seasonal/var_total},
    {'Component': 'Residual', 'Variance': var_residual, 'Contribution (%)': 100*var_residual/var_total},
    {'Component': 'Total', 'Variance': var_total, 'Contribution (%)': 100.0}
])

# Display with styling
display(Markdown("### STL Decomposition - Variance Contribution"))

styled_variance = variance_df.style\
    .format({'Variance': '{:.4f}', 'Contribution (%)': '{:.2f}%'})\
    .background_gradient(subset=['Contribution (%)'], cmap='RdYlGn', vmin=0, vmax=100)\
    .applymap(lambda x: 'font-weight: bold' if x == 'Total' else '', subset=['Component'])

display(styled_variance)

In [ ]:
# Plot STL decomposition
fig, axes = plt.subplots(4, 1, figsize=(14, 12))

# Original
axes[0].plot(df_stl.index, df_stl.values, color='blue', linewidth=0.8)
axes[0].set_ylabel('Original', fontweight='bold')
axes[0].set_title('Décomposition STL - Piezo1 Level', fontsize=14, fontweight='bold', pad=20)
axes[0].grid(True, alpha=0.3)

# Trend
axes[1].plot(trend.index, trend.values, color='red', linewidth=1.2)
axes[1].set_ylabel('Tendance', fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].text(0.02, 0.95, f'Variance: {100*var_trend/var_total:.1f}%', 
             transform=axes[1].transAxes, fontsize=10, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Seasonal
axes[2].plot(seasonal.index, seasonal.values, color='green', linewidth=0.8)
axes[2].set_ylabel('Saisonnalité', fontweight='bold')
axes[2].grid(True, alpha=0.3)
axes[2].text(0.02, 0.95, f'Variance: {100*var_seasonal/var_total:.1f}%', 
             transform=axes[2].transAxes, fontsize=10, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Residual
axes[3].plot(residual.index, residual.values, color='purple', linewidth=0.6, alpha=0.7)
axes[3].set_ylabel('Résidus', fontweight='bold')
axes[3].set_xlabel('Date', fontweight='bold')
axes[3].grid(True, alpha=0.3)
axes[3].text(0.02, 0.95, f'Variance: {100*var_residual/var_total:.1f}%', 
             transform=axes[3].transAxes, fontsize=10, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig(figs_dir / 'eda_piezo1_stl_decomposition.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. ACF/PACF (Piezo1)

Autocorrélation et autocorrélation partielle pour identifier les dépendances temporelles.

In [ ]:
# ACF and PACF for level
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Level - ACF
plot_acf(df_piezo1['level'].dropna(), lags=100, ax=axes[0, 0], alpha=0.05)
axes[0, 0].set_title('ACF - Niveau Piézométrique', fontweight='bold')
axes[0, 0].set_xlabel('Lag (jours)')

# Level - PACF
plot_pacf(df_piezo1['level'].dropna(), lags=100, ax=axes[0, 1], alpha=0.05)
axes[0, 1].set_title('PACF - Niveau Piézométrique', fontweight='bold')
axes[0, 1].set_xlabel('Lag (jours)')

# Precipitation - ACF
plot_acf(df_piezo1['PRELIQ_Q'].dropna(), lags=100, ax=axes[1, 0], alpha=0.05)
axes[1, 0].set_title('ACF - Précipitation', fontweight='bold')
axes[1, 0].set_xlabel('Lag (jours)')

# Precipitation - PACF
plot_pacf(df_piezo1['PRELIQ_Q'].dropna(), lags=100, ax=axes[1, 1], alpha=0.05)
axes[1, 1].set_title('PACF - Précipitation', fontweight='bold')
axes[1, 1].set_xlabel('Lag (jours)')

plt.suptitle('Autocorrélation et Autocorrélation Partielle - Piezo1', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(figs_dir / 'eda_piezo1_acf_pacf.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Cross-Correlation (Piezo1)

Analyse de corrélation croisée entre précipitation et niveau piézométrique pour identifier le lag optimal.

In [ ]:
def cross_correlation(x, y, max_lag=60):
    """
    Calculate cross-correlation between two series
    Returns correlations and lags
    """
    correlations = []
    lags = range(-max_lag, max_lag + 1)
    
    for lag in lags:
        if lag < 0:
            corr = np.corrcoef(x[:lag], y[-lag:])[0, 1]
        elif lag > 0:
            corr = np.corrcoef(x[lag:], y[:-lag])[0, 1]
        else:
            corr = np.corrcoef(x, y)[0, 1]
        correlations.append(corr)
    
    return lags, correlations

# Calculate cross-correlation
lags, ccf = cross_correlation(
    df_piezo1['PRELIQ_Q'].values,
    df_piezo1['level'].values,
    max_lag=60
)

# Find optimal lag
optimal_lag_idx = np.argmax(np.abs(ccf))
optimal_lag = lags[optimal_lag_idx]
optimal_corr = ccf[optimal_lag_idx]

# Create summary DataFrame
ccf_summary = pd.DataFrame([{
    'Metric': 'Optimal Lag',
    'Value': f'{optimal_lag} days'
}, {
    'Metric': 'Maximum Correlation',
    'Value': f'{optimal_corr:.4f}'
}, {
    'Metric': 'Direction',
    'Value': 'PRELIQ leads level' if optimal_lag > 0 else 'Level leads PRELIQ'
}])

display(Markdown("### Cross-Correlation Analysis: PRELIQ_Q → Level"))
display(ccf_summary.style.hide(axis='index').set_properties(**{'text-align': 'left'}))

# Plot cross-correlation
fig, ax = plt.subplots(figsize=(14, 6))

ax.stem(lags, ccf, linefmt='blue', markerfmt='o', basefmt=' ')
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.8)

# Confidence bounds (approximate)
conf_bound = 1.96 / np.sqrt(len(df_piezo1))
ax.axhline(y=conf_bound, color='red', linestyle='--', linewidth=1, alpha=0.7, label='Confidence bounds')
ax.axhline(y=-conf_bound, color='red', linestyle='--', linewidth=1, alpha=0.7)

# Mark optimal lag
ax.scatter([optimal_lag], [optimal_corr], color='red', s=100, zorder=5, label=f'Optimal lag: {optimal_lag} days')

ax.set_xlabel('Lag (jours)', fontweight='bold')
ax.set_ylabel('Corrélation', fontweight='bold')
ax.set_title('Corrélation Croisée: Précipitation → Niveau Piézométrique', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
plt.savefig(figs_dir / 'eda_piezo1_cross_correlation.png', dpi=300, bbox_inches='tight')
plt.show()

## 9. Granger Causality (Piezo1)

Tests de causalité de Granger pour déterminer si les covariables aident à prédire le niveau piézométrique.

**H₀**: La covariable ne cause PAS au sens de Granger la variable cible (rejeter si p < 0.05)

In [ ]:
def granger_causality_analysis(df, target_col, covariate_col, max_lag=30):
    """
    Perform Granger causality test
    H0: covariate does NOT Granger-cause target
    """
    # Prepare data
    data = df[[covariate_col, target_col]].dropna()
    
    # Run test
    try:
        gc_result = grangercausalitytests(data, maxlag=max_lag, verbose=False)
        
        # Extract p-values for different lags
        lags = []
        pvalues = []
        
        for lag in range(1, max_lag + 1):
            # Get F-test p-value
            pvalue = gc_result[lag][0]['ssr_ftest'][1]
            lags.append(lag)
            pvalues.append(pvalue)
        
        # Find significant lags (p < 0.05)
        significant_lags = [lag for lag, pval in zip(lags, pvalues) if pval < 0.05]
        
        return lags, pvalues, significant_lags
    
    except Exception as e:
        return [], [], []

# Test Granger causality for each covariate
covariates = ['PRELIQ_Q', 'T_Q', 'ETP_Q']
granger_results = {}
granger_summary_data = []

for cov in covariates:
    lags, pvalues, sig_lags = granger_causality_analysis(
        df_piezo1, 'level', cov, max_lag=30
    )
    granger_results[cov] = {'lags': lags, 'pvalues': pvalues, 'significant_lags': sig_lags}
    
    # Create summary entry
    if sig_lags:
        best_lag = sig_lags[0]
        best_pvalue = pvalues[best_lag - 1]
        sig_lags_str = ', '.join(map(str, sig_lags[:5])) + ('...' if len(sig_lags) > 5 else '')
    else:
        best_lag = np.nan
        best_pvalue = np.nan
        sig_lags_str = 'None'
    
    granger_summary_data.append({
        'Covariate': cov,
        'Significant Lags': sig_lags_str,
        'Best Lag': best_lag,
        'P-value': best_pvalue,
        'Granger Causality': 'Yes' if sig_lags else 'No'
    })

# Create summary DataFrame
granger_summary_df = pd.DataFrame(granger_summary_data)

# Display with styling
display(Markdown("### Granger Causality Test Results"))

def color_causality(val):
    if val == 'Yes':
        return 'background-color: #90EE90; font-weight: bold'
    elif val == 'No':
        return 'background-color: #FFB6C6'
    return ''

styled_granger = granger_summary_df.style\
    .format({'P-value': '{:.6f}', 'Best Lag': '{:.0f}'}, na_rep='—')\
    .applymap(color_causality, subset=['Granger Causality'])\
    .applymap(lambda x: 'background-color: #90EE90' if isinstance(x, (int, float)) and x < 0.05 else '', subset=['P-value'])

display(styled_granger)

In [ ]:
# Plot Granger causality p-values
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

cov_names = {
    'PRELIQ_Q': 'Précipitation',
    'T_Q': 'Température',
    'ETP_Q': 'Évapotranspiration'
}

for idx, cov in enumerate(covariates):
    if granger_results[cov]['lags']:
        lags = granger_results[cov]['lags']
        pvalues = granger_results[cov]['pvalues']
        
        axes[idx].plot(lags, pvalues, marker='o', linewidth=2, markersize=4)
        axes[idx].axhline(y=0.05, color='red', linestyle='--', linewidth=2, label='α = 0.05')
        axes[idx].set_xlabel('Lag (jours)', fontweight='bold')
        axes[idx].set_ylabel('P-value', fontweight='bold')
        axes[idx].set_title(f'Granger Causality\n{cov_names[cov]} → Level', fontweight='bold')
        axes[idx].grid(True, alpha=0.3)
        axes[idx].legend()
        axes[idx].set_ylim(-0.05, 1.05)

plt.suptitle('Tests de Causalité de Granger - Piezo1', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(figs_dir / 'eda_piezo1_granger_causality.png', dpi=300, bbox_inches='tight')
plt.show()

## 10. Lag Analysis (Piezo1)

Analyse détaillée des corrélations pour différents lags entre covariables et niveau piézométrique.

In [ ]:
def calculate_lagged_correlations(df, target_col, covariate_col, max_lag=60):
    """
    Calculate correlation between lagged covariate and target
    """
    correlations = []
    lags = range(0, max_lag + 1)
    
    for lag in lags:
        # Shift covariate backward by lag
        df_temp = df[[target_col, covariate_col]].copy()
        df_temp['covariate_lagged'] = df_temp[covariate_col].shift(lag)
        df_temp = df_temp.dropna()
        
        if len(df_temp) > 0:
            corr = df_temp[[target_col, 'covariate_lagged']].corr().iloc[0, 1]
            correlations.append(corr)
        else:
            correlations.append(np.nan)
    
    return list(lags), correlations

# Calculate for each covariate
lag_analysis_results = {}
lag_summary_data = []

cov_names = {
    'PRELIQ_Q': 'Précipitation',
    'T_Q': 'Température',
    'ETP_Q': 'Évapotranspiration'
}

for cov in covariates:
    lags, corrs = calculate_lagged_correlations(df_piezo1, 'level', cov, max_lag=60)
    optimal_lag_idx = np.nanargmax(np.abs(corrs))
    optimal_lag = lags[optimal_lag_idx]
    optimal_corr = corrs[optimal_lag_idx]
    
    lag_analysis_results[cov] = {
        'lags': lags,
        'correlations': corrs,
        'optimal_lag': optimal_lag,
        'optimal_correlation': optimal_corr
    }
    
    lag_summary_data.append({
        'Covariate': cov_names[cov],
        'Optimal Lag (days)': optimal_lag,
        'Max Correlation': optimal_corr
    })

# Create summary DataFrame
lag_summary_df = pd.DataFrame(lag_summary_data)

# Display with styling
display(Markdown("### Lag Analysis Results"))

styled_lag = lag_summary_df.style\
    .format({'Optimal Lag (days)': '{:.0f}', 'Max Correlation': '{:.4f}'})\
    .background_gradient(subset=['Max Correlation'], cmap='RdYlGn', vmin=-1, vmax=1)\
    .highlight_max(subset=['Max Correlation'], color='#90EE90')

display(styled_lag)

In [ ]:
# Plot lag correlations
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, cov in enumerate(covariates):
    lags = lag_analysis_results[cov]['lags']
    corrs = lag_analysis_results[cov]['correlations']
    opt_lag = lag_analysis_results[cov]['optimal_lag']
    opt_corr = lag_analysis_results[cov]['optimal_correlation']
    
    axes[idx].plot(lags, corrs, linewidth=2, color='blue')
    axes[idx].scatter([opt_lag], [opt_corr], color='red', s=100, zorder=5, 
                      label=f'Optimal: {opt_lag} days')
    axes[idx].axhline(y=0, color='black', linestyle='-', linewidth=0.8)
    axes[idx].set_xlabel('Lag (jours)', fontweight='bold')
    axes[idx].set_ylabel('Corrélation', fontweight='bold')
    axes[idx].set_title(f'{cov_names[cov]}\nvs Level', fontweight='bold')
    axes[idx].grid(True, alpha=0.3)
    axes[idx].legend()

plt.suptitle('Analyse de Lag - Corrélations avec Niveau Piézométrique', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(figs_dir / 'eda_piezo1_lag_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

## 11. Distributions (Piezo1)

Analyse de la distribution des variables avec tests de normalité.

In [ ]:
# Distribution analysis
fig, axes = plt.subplots(4, 2, figsize=(14, 14))

variables_full = [
    ('level', 'Niveau Piézométrique'),
    ('PRELIQ_Q', 'Précipitation'),
    ('T_Q', 'Température'),
    ('ETP_Q', 'Évapotranspiration')
]

normality_results = []

for idx, (var, label) in enumerate(variables_full):
    data = df_piezo1[var].dropna()
    
    # Histogram
    axes[idx, 0].hist(data, bins=50, edgecolor='black', alpha=0.7)
    axes[idx, 0].set_xlabel(label, fontweight='bold')
    axes[idx, 0].set_ylabel('Fréquence', fontweight='bold')
    axes[idx, 0].set_title(f'Distribution - {label}', fontweight='bold')
    axes[idx, 0].grid(True, alpha=0.3)
    
    # Q-Q plot
    stats.probplot(data, dist="norm", plot=axes[idx, 1])
    axes[idx, 1].set_title(f'Q-Q Plot - {label}', fontweight='bold')
    axes[idx, 1].grid(True, alpha=0.3)
    
    # Shapiro-Wilk test
    if len(data) <= 5000:  # Shapiro-Wilk has sample size limitations
        shapiro_stat, shapiro_pvalue = stats.shapiro(data)
    else:
        # Use subsample for large datasets
        sample_data = np.random.choice(data, 5000, replace=False)
        shapiro_stat, shapiro_pvalue = stats.shapiro(sample_data)
    
    normality_results.append({
        'Variable': label,
        'Shapiro-Wilk Stat': shapiro_stat,
        'P-value': shapiro_pvalue,
        'Normal Distribution': 'Yes' if shapiro_pvalue > 0.05 else 'No'
    })

plt.suptitle('Distribution des Variables - Piezo1', fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig(figs_dir / 'eda_piezo1_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

# Display normality test results
normality_df = pd.DataFrame(normality_results)

display(Markdown("### Normality Tests (Shapiro-Wilk)"))

def color_normality(val):
    if val == 'Yes':
        return 'background-color: #90EE90; font-weight: bold'
    elif val == 'No':
        return 'background-color: #FFB6C6; font-weight: bold'
    return ''

styled_normality = normality_df.style\
    .format({'Shapiro-Wilk Stat': '{:.4f}', 'P-value': '{:.4f}'})\
    .applymap(lambda x: 'background-color: #90EE90' if isinstance(x, (int, float)) and x > 0.05 else 'background-color: #FFB6C6', subset=['P-value'])\
    .applymap(color_normality, subset=['Normal Distribution'])

display(styled_normality)

## 12. Residuals Analysis (Piezo1)

Analyse détaillée des résidus de la décomposition STL.

In [ ]:
# Residuals analysis from STL decomposition
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Time series plot
axes[0, 0].plot(residual.index, residual.values, linewidth=0.6, alpha=0.7)
axes[0, 0].axhline(y=0, color='red', linestyle='--', linewidth=1)
axes[0, 0].set_xlabel('Date', fontweight='bold')
axes[0, 0].set_ylabel('Résidus', fontweight='bold')
axes[0, 0].set_title('Résidus STL dans le Temps', fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# Histogram
axes[0, 1].hist(residual.dropna(), bins=50, edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Résidus', fontweight='bold')
axes[0, 1].set_ylabel('Fréquence', fontweight='bold')
axes[0, 1].set_title('Distribution des Résidus', fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# Q-Q plot
stats.probplot(residual.dropna(), dist="norm", plot=axes[1, 0])
axes[1, 0].set_title('Q-Q Plot des Résidus', fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# ACF of residuals
plot_acf(residual.dropna(), lags=100, ax=axes[1, 1], alpha=0.05)
axes[1, 1].set_title('ACF des Résidus', fontweight='bold')
axes[1, 1].set_xlabel('Lag (jours)')

plt.suptitle('Analyse des Résidus - Décomposition STL Piezo1', 
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig(figs_dir / 'eda_piezo1_residuals.png', dpi=300, bbox_inches='tight')
plt.show()

# Ljung-Box test for white noise
lb_test = acorr_ljungbox(residual.dropna(), lags=20, return_df=True)

# Display Ljung-Box results
display(Markdown("### Ljung-Box Test for White Noise"))
display(Markdown("**H₀:** Les résidus sont un bruit blanc (pas de corrélation)"))

lb_display = lb_test[['lb_stat', 'lb_pvalue']].head(10).copy()
lb_display.columns = ['Ljung-Box Stat', 'P-value']
lb_display['White Noise'] = lb_display['P-value'].apply(lambda x: 'Yes' if x > 0.05 else 'No')

def color_white_noise(val):
    if val == 'Yes':
        return 'background-color: #90EE90'
    elif val == 'No':
        return 'background-color: #FFB6C6'
    return ''

styled_lb = lb_display.style\
    .format({'Ljung-Box Stat': '{:.4f}', 'P-value': '{:.4f}'})\
    .applymap(lambda x: 'background-color: #90EE90' if isinstance(x, (int, float)) and x > 0.05 else 'background-color: #FFB6C6', subset=['P-value'])\
    .applymap(color_white_noise, subset=['White Noise'])

display(styled_lb)

# Interpretation
if (lb_test['lb_pvalue'] < 0.05).any():
    display(Markdown("**Interpretation:** Résidus correlés détectés (rejeter H₀) - Il existe des dépendances temporelles dans les résidus"))
else:
    display(Markdown("**Interpretation:** Résidus = bruit blanc (ne pas rejeter H₀) - Pas de corrélation significative détectée"))

## 13. Correlation Matrix (Piezo1)

Matrice de corrélation entre toutes les variables.

In [ ]:
# Correlation matrix
corr_matrix = df_piezo1[['level', 'PRELIQ_Q', 'T_Q', 'ETP_Q']].corr()

fig, ax = plt.subplots(figsize=(10, 8))

sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm', 
            center=0, vmin=-1, vmax=1, square=True, 
            linewidths=1, cbar_kws={"shrink": 0.8}, ax=ax)

ax.set_title('Matrice de Corrélation - Piezo1', fontsize=14, fontweight='bold', pad=20)

# Better labels
labels = ['Niveau', 'Précipitation', 'Température', 'ETP']
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_yticklabels(labels, rotation=0)

plt.tight_layout()
plt.savefig(figs_dir / 'eda_piezo1_correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

# Display as styled DataFrame
display(Markdown("### Correlation Matrix"))

styled_corr = corr_matrix.style\
    .format('{:.3f}')\
    .background_gradient(cmap='coolwarm', vmin=-1, vmax=1, axis=None)

display(styled_corr)

## 14. Scatter Plots (Piezo1)

Diagrammes de dispersion entre niveau piézométrique et covariables.

In [ ]:
# Scatter plots
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

scatter_vars = [
    ('PRELIQ_Q', 'Précipitation (mm)'),
    ('T_Q', 'Température (°C)'),
    ('ETP_Q', 'Évapotranspiration (mm)')
]

for idx, (var, label) in enumerate(scatter_vars):
    # Scatter plot with density
    axes[idx].scatter(df_piezo1[var], df_piezo1['level'], 
                     alpha=0.3, s=10, edgecolors='none')
    
    # Add regression line
    z = np.polyfit(df_piezo1[var].dropna(), 
                   df_piezo1.loc[df_piezo1[var].notna(), 'level'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(df_piezo1[var].min(), df_piezo1[var].max(), 100)
    axes[idx].plot(x_line, p(x_line), "r-", linewidth=2, label='Regression')
    
    # Calculate correlation
    corr = df_piezo1[[var, 'level']].corr().iloc[0, 1]
    
    axes[idx].set_xlabel(label, fontweight='bold')
    axes[idx].set_ylabel('Niveau Piézométrique (m)', fontweight='bold')
    axes[idx].set_title(f'{label} vs Niveau\n(r = {corr:.3f})', fontweight='bold')
    axes[idx].grid(True, alpha=0.3)
    axes[idx].legend()

plt.suptitle('Scatter Plots - Piezo1', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(figs_dir / 'eda_piezo1_scatter_plots.png', dpi=300, bbox_inches='tight')
plt.show()

## 15. Monthly Boxplots (Piezo1)

Boîtes à moustaches par mois pour visualiser les patterns saisonniers.

In [ ]:
# Add month column
df_piezo1['month'] = df_piezo1['date'].dt.month

# Monthly boxplots
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

boxplot_vars = [
    ('level', 'Niveau Piézométrique (m)'),
    ('PRELIQ_Q', 'Précipitation (mm)'),
    ('T_Q', 'Température (°C)'),
    ('ETP_Q', 'Évapotranspiration (mm)')
]

month_names = ['Jan', 'Fév', 'Mar', 'Avr', 'Mai', 'Jun', 
               'Jul', 'Aoû', 'Sep', 'Oct', 'Nov', 'Déc']

for idx, (var, label) in enumerate(boxplot_vars):
    data_by_month = [df_piezo1[df_piezo1['month'] == m][var].dropna().values 
                     for m in range(1, 13)]
    
    bp = axes[idx].boxplot(data_by_month, labels=month_names, patch_artist=True)
    
    # Color boxes
    for patch in bp['boxes']:
        patch.set_facecolor('lightblue')
        patch.set_alpha(0.7)
    
    axes[idx].set_xlabel('Mois', fontweight='bold')
    axes[idx].set_ylabel(label, fontweight='bold')
    axes[idx].set_title(f'Distribution Mensuelle - {label.split("(")[0].strip()}', 
                        fontweight='bold')
    axes[idx].grid(True, alpha=0.3, axis='y')
    axes[idx].tick_params(axis='x', rotation=45)

plt.suptitle('Patterns Saisonniers Mensuels - Piezo1', 
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig(figs_dir / 'eda_piezo1_monthly_boxplots.png', dpi=300, bbox_inches='tight')
plt.show()

## 16. AGGREGATE ANALYSIS ACROSS ALL 18 STATIONS

Analyse comparative de toutes les stations pour identifier les patterns communs et les outliers.

In [ ]:
display(Markdown("### Processing All 18 Stations..."))

# Store results for all stations
all_stations_results = []

for station_name, df_station in stations.items():
    # Clean data
    df_clean = df_station.dropna()
    
    if len(df_clean) < 100:  # Skip if too few samples
        continue
    
    # 1. Stationarity tests
    try:
        adf_result = adfuller(df_clean['level'], autolag='AIC')
        adf_pvalue = adf_result[1]
        adf_stationary = adf_pvalue < 0.05
    except:
        adf_pvalue = np.nan
        adf_stationary = False
    
    try:
        kpss_result = kpss(df_clean['level'], regression='ct', nlags='auto')
        kpss_pvalue = kpss_result[1]
        kpss_stationary = kpss_pvalue >= 0.05
    except:
        kpss_pvalue = np.nan
        kpss_stationary = False
    
    # 2. Correlations with level
    corr_preliq = df_clean[['level', 'PRELIQ_Q']].corr().iloc[0, 1]
    corr_temp = df_clean[['level', 'T_Q']].corr().iloc[0, 1]
    corr_etp = df_clean[['level', 'ETP_Q']].corr().iloc[0, 1]
    
    # 3. Optimal lag for precipitation
    try:
        lags, corrs = calculate_lagged_correlations(df_clean, 'level', 'PRELIQ_Q', max_lag=60)
        optimal_lag_idx = np.nanargmax(np.abs(corrs))
        optimal_lag = lags[optimal_lag_idx]
        optimal_lag_corr = corrs[optimal_lag_idx]
    except:
        optimal_lag = np.nan
        optimal_lag_corr = np.nan
    
    # 4. Seasonality detection
    try:
        ts_station = TimeSeries.from_dataframe(
            df_clean[['date', 'level']].set_index('date'),
            value_cols='level'
        )
        seasonal_365 = check_seasonality(ts_station, m=365, max_lag=730, alpha=0.05)
    except:
        seasonal_365 = False
    
    # Store results
    all_stations_results.append({
        'Station': station_name,
        'Samples': len(df_clean),
        'ADF p-value': adf_pvalue,
        'ADF Stationary': adf_stationary,
        'KPSS p-value': kpss_pvalue,
        'KPSS Stationary': kpss_stationary,
        'Corr PRELIQ': corr_preliq,
        'Corr T': corr_temp,
        'Corr ETP': corr_etp,
        'Optimal Lag': optimal_lag,
        'Lag Correlation': optimal_lag_corr,
        'Seasonal (365d)': seasonal_365
    })

# Create summary DataFrame
aggregate_df = pd.DataFrame(all_stations_results)

# Display summary
display(Markdown("### Summary Table - All Stations"))

styled_aggregate = aggregate_df.style\
    .format({
        'Samples': '{:,}',
        'ADF p-value': '{:.4f}',
        'KPSS p-value': '{:.4f}',
        'Corr PRELIQ': '{:.3f}',
        'Corr T': '{:.3f}',
        'Corr ETP': '{:.3f}',
        'Optimal Lag': '{:.0f}',
        'Lag Correlation': '{:.3f}'
    }, na_rep='—')\
    .applymap(lambda x: 'background-color: #90EE90' if x else 'background-color: #FFB6C6', subset=['ADF Stationary', 'KPSS Stationary', 'Seasonal (365d)'])\
    .background_gradient(subset=['Corr PRELIQ'], cmap='RdYlGn', vmin=-1, vmax=1)\
    .background_gradient(subset=['Optimal Lag'], cmap='YlOrRd')

display(styled_aggregate)

In [ ]:
# Create aggregate statistics summary
aggregate_stats = pd.DataFrame([
    {
        'Metric': 'Total Stations',
        'Value': f"{len(aggregate_df)}"
    },
    {
        'Metric': 'ADF Stationary Stations',
        'Value': f"{aggregate_df['ADF Stationary'].sum()} / {len(aggregate_df)}"
    },
    {
        'Metric': 'KPSS Stationary Stations',
        'Value': f"{aggregate_df['KPSS Stationary'].sum()} / {len(aggregate_df)}"
    },
    {
        'Metric': 'Seasonal Stations (365d)',
        'Value': f"{aggregate_df['Seasonal (365d)'].sum()} / {len(aggregate_df)}"
    },
    {
        'Metric': 'Mean Correlation (PRELIQ_Q)',
        'Value': f"{aggregate_df['Corr PRELIQ'].mean():.3f} ± {aggregate_df['Corr PRELIQ'].std():.3f}"
    },
    {
        'Metric': 'Mean Correlation (T_Q)',
        'Value': f"{aggregate_df['Corr T'].mean():.3f} ± {aggregate_df['Corr T'].std():.3f}"
    },
    {
        'Metric': 'Mean Correlation (ETP_Q)',
        'Value': f"{aggregate_df['Corr ETP'].mean():.3f} ± {aggregate_df['Corr ETP'].std():.3f}"
    },
    {
        'Metric': 'Mean Optimal Lag',
        'Value': f"{aggregate_df['Optimal Lag'].mean():.1f} ± {aggregate_df['Optimal Lag'].std():.1f} days"
    },
    {
        'Metric': 'Optimal Lag Range',
        'Value': f"{aggregate_df['Optimal Lag'].min():.0f} - {aggregate_df['Optimal Lag'].max():.0f} days"
    },
    {
        'Metric': 'Median Optimal Lag',
        'Value': f"{aggregate_df['Optimal Lag'].median():.0f} days"
    }
])

display(Markdown("### Aggregate Statistics"))
display(aggregate_stats.style.hide(axis='index').set_properties(**{'text-align': 'left'}))

# Visualize aggregate results
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Distribution of optimal lags
axes[0, 0].hist(aggregate_df['Optimal Lag'].dropna(), bins=15, edgecolor='black', alpha=0.7)
axes[0, 0].axvline(aggregate_df['Optimal Lag'].mean(), color='red', 
                   linestyle='--', linewidth=2, label=f'Mean: {aggregate_df["Optimal Lag"].mean():.1f} days')
axes[0, 0].set_xlabel('Lag Optimal (jours)', fontweight='bold')
axes[0, 0].set_ylabel('Nombre de Stations', fontweight='bold')
axes[0, 0].set_title('Distribution des Lags Optimaux', fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend()

# 2. Mean correlation by station
stations_sorted = aggregate_df.sort_values('Corr PRELIQ')
x_pos = np.arange(len(stations_sorted))
axes[0, 1].barh(x_pos, stations_sorted['Corr PRELIQ'].abs(), alpha=0.7)
axes[0, 1].set_yticks(x_pos)
axes[0, 1].set_yticklabels(stations_sorted['Station'], fontsize=8)
axes[0, 1].set_xlabel('|Corrélation| PRELIQ_Q vs Level', fontweight='bold')
axes[0, 1].set_title('Corrélation par Station', fontweight='bold')
axes[0, 1].grid(True, alpha=0.3, axis='x')

# 3. Correlation comparison
corr_means = [
    aggregate_df['Corr PRELIQ'].mean(),
    aggregate_df['Corr T'].mean(),
    aggregate_df['Corr ETP'].mean()
]
corr_stds = [
    aggregate_df['Corr PRELIQ'].std(),
    aggregate_df['Corr T'].std(),
    aggregate_df['Corr ETP'].std()
]
x_labels = ['PRELIQ_Q', 'T_Q', 'ETP_Q']
x_pos2 = np.arange(len(x_labels))
axes[1, 0].bar(x_pos2, corr_means, yerr=corr_stds, capsize=10, alpha=0.7, edgecolor='black')
axes[1, 0].set_xticks(x_pos2)
axes[1, 0].set_xticklabels(x_labels)
axes[1, 0].set_ylabel('Corrélation Moyenne', fontweight='bold')
axes[1, 0].set_title('Corrélations Moyennes (±std) - Toutes Stations', fontweight='bold')
axes[1, 0].grid(True, alpha=0.3, axis='y')
axes[1, 0].axhline(y=0, color='black', linestyle='-', linewidth=0.8)

# 4. Stationarity and seasonality summary
summary_data = [
    aggregate_df['ADF Stationary'].sum(),
    aggregate_df['KPSS Stationary'].sum(),
    aggregate_df['Seasonal (365d)'].sum()
]
summary_labels = ['ADF\nStationary', 'KPSS\nStationary', 'Seasonal\n(365d)']
x_pos3 = np.arange(len(summary_labels))
bars = axes[1, 1].bar(x_pos3, summary_data, alpha=0.7, edgecolor='black')
bars[0].set_color('green')
bars[1].set_color('blue')
bars[2].set_color('orange')
axes[1, 1].set_xticks(x_pos3)
axes[1, 1].set_xticklabels(summary_labels)
axes[1, 1].set_ylabel('Nombre de Stations', fontweight='bold')
axes[1, 1].set_title(f'Tests de Stationnarité et Saisonnalité (n={len(aggregate_df)})', 
                     fontweight='bold')
axes[1, 1].set_ylim(0, len(aggregate_df) + 1)
axes[1, 1].grid(True, alpha=0.3, axis='y')

# Add values on bars
for bar in bars:
    height = bar.get_height()
    axes[1, 1].text(bar.get_x() + bar.get_width()/2., height,
                    f'{int(height)}',
                    ha='center', va='bottom', fontweight='bold', fontsize=12)

plt.suptitle('Analyse Agrégée - 18 Stations Piézométriques', 
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig(figs_dir / 'eda_aggregate_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Identify outlier stations
display(Markdown("### Outlier Detection"))

# Stations with unusual optimal lags
mean_lag = aggregate_df['Optimal Lag'].mean()
std_lag = aggregate_df['Optimal Lag'].std()
outlier_lag = aggregate_df[
    (aggregate_df['Optimal Lag'] < mean_lag - 2*std_lag) | 
    (aggregate_df['Optimal Lag'] > mean_lag + 2*std_lag)
]

if len(outlier_lag) > 0:
    display(Markdown("#### Stations with Unusual Optimal Lags (>2σ from mean)"))
    styled_outlier_lag = outlier_lag[['Station', 'Optimal Lag', 'Lag Correlation']].style\
        .format({'Optimal Lag': '{:.0f}', 'Lag Correlation': '{:.4f}'})\
        .background_gradient(subset=['Optimal Lag'], cmap='Reds')
    display(styled_outlier_lag)
else:
    display(Markdown("**No stations with unusual optimal lags detected.**"))

# Stations with very low correlation with precipitation
low_corr_threshold = aggregate_df['Corr PRELIQ'].quantile(0.25)
low_corr_stations = aggregate_df[aggregate_df['Corr PRELIQ'].abs() < low_corr_threshold]

display(Markdown(f"#### Stations with Low Correlation with PRELIQ_Q (< Q1 = {low_corr_threshold:.3f})"))
styled_low_corr = low_corr_stations[['Station', 'Corr PRELIQ', 'Optimal Lag']].style\
    .format({'Corr PRELIQ': '{:.3f}', 'Optimal Lag': '{:.0f}'}, na_rep='—')\
    .background_gradient(subset=['Corr PRELIQ'], cmap='RdYlGn', vmin=-1, vmax=1)
display(styled_low_corr)

# Non-stationary stations
non_stationary = aggregate_df[
    (~aggregate_df['ADF Stationary']) & (~aggregate_df['KPSS Stationary'])
]

if len(non_stationary) > 0:
    display(Markdown("#### Non-Stationary Stations (Both Tests)"))
    styled_non_stat = non_stationary[['Station', 'ADF p-value', 'KPSS p-value']].style\
        .format({'ADF p-value': '{:.4f}', 'KPSS p-value': '{:.4f}'}, na_rep='—')\
        .applymap(lambda x: 'background-color: #FFB6C6' if isinstance(x, (int, float)) else '')
    display(styled_non_stat)
else:
    display(Markdown("**All stations show some stationarity.**"))

## 17. Summary & Recommendations

Synthèse des résultats et recommandations pour la modélisation.

In [ ]:
display(Markdown("# Modeling Recommendations"))

# 1. Key Findings from Piezo1
piezo1_results = aggregate_df[aggregate_df['Station'] == 'piezo1'].iloc[0]

piezo1_summary = pd.DataFrame([
    {'Category': 'Data', 'Metric': 'Total Samples', 'Value': f"{len(df_piezo1):,}"},
    {'Category': 'Data', 'Metric': 'Date Range', 'Value': f"{df_piezo1['date'].min().date()} to {df_piezo1['date'].max().date()}"},
    {'Category': 'Data', 'Metric': 'Duration', 'Value': f"{(df_piezo1['date'].max() - df_piezo1['date'].min()).days:,} days"},
    {'Category': 'Stationarity', 'Metric': 'ADF Test', 'Value': 'STATIONARY' if piezo1_results['ADF Stationary'] else 'NON-STATIONARY'},
    {'Category': 'Stationarity', 'Metric': 'KPSS Test', 'Value': 'STATIONARY' if piezo1_results['KPSS Stationary'] else 'NON-STATIONARY'},
    {'Category': 'Seasonality', 'Metric': 'Annual (365d)', 'Value': 'DETECTED' if piezo1_results['Seasonal (365d)'] else 'NOT DETECTED'},
    {'Category': 'Decomposition', 'Metric': 'Trend Variance', 'Value': f"{100*var_trend/var_total:.1f}%"},
    {'Category': 'Decomposition', 'Metric': 'Seasonal Variance', 'Value': f"{100*var_seasonal/var_total:.1f}%"},
    {'Category': 'Correlation', 'Metric': 'PRELIQ_Q', 'Value': f"{piezo1_results['Corr PRELIQ']:.3f}"},
    {'Category': 'Correlation', 'Metric': 'T_Q', 'Value': f"{piezo1_results['Corr T']:.3f}"},
    {'Category': 'Correlation', 'Metric': 'ETP_Q', 'Value': f"{piezo1_results['Corr ETP']:.3f}"},
    {'Category': 'Lag Analysis', 'Metric': 'Optimal Lag (PRELIQ_Q)', 'Value': f"{piezo1_results['Optimal Lag']:.0f} days"},
    {'Category': 'Lag Analysis', 'Metric': 'Correlation at Optimal Lag', 'Value': f"{piezo1_results['Lag Correlation']:.3f}"}
])

display(Markdown("### 1. Key Findings from Piezo1 (Representative Station)"))
display(piezo1_summary.style.hide(axis='index').set_properties(**{'text-align': 'left'}))

# 2. Variability Across All Stations
variability_summary = pd.DataFrame([
    {'Metric': 'Total Stations Analyzed', 'Value': f"{len(aggregate_df)}"},
    {'Metric': 'ADF Stationary Stations', 'Value': f"{aggregate_df['ADF Stationary'].sum()} / {len(aggregate_df)} ({100*aggregate_df['ADF Stationary'].sum()/len(aggregate_df):.0f}%)"},
    {'Metric': 'KPSS Stationary Stations', 'Value': f"{aggregate_df['KPSS Stationary'].sum()} / {len(aggregate_df)} ({100*aggregate_df['KPSS Stationary'].sum()/len(aggregate_df):.0f}%)"},
    {'Metric': 'Seasonal Stations (365d)', 'Value': f"{aggregate_df['Seasonal (365d)'].sum()} / {len(aggregate_df)} ({100*aggregate_df['Seasonal (365d)'].sum()/len(aggregate_df):.0f}%)"},
    {'Metric': 'Mean Correlation (PRELIQ_Q)', 'Value': f"{aggregate_df['Corr PRELIQ'].mean():.3f} ± {aggregate_df['Corr PRELIQ'].std():.3f}"},
    {'Metric': 'Mean Correlation (T_Q)', 'Value': f"{aggregate_df['Corr T'].mean():.3f} ± {aggregate_df['Corr T'].std():.3f}"},
    {'Metric': 'Mean Correlation (ETP_Q)', 'Value': f"{aggregate_df['Corr ETP'].mean():.3f} ± {aggregate_df['Corr ETP'].std():.3f}"},
    {'Metric': 'Mean Optimal Lag', 'Value': f"{aggregate_df['Optimal Lag'].mean():.1f} ± {aggregate_df['Optimal Lag'].std():.1f} days"},
    {'Metric': 'Median Optimal Lag', 'Value': f"{aggregate_df['Optimal Lag'].median():.0f} days"},
    {'Metric': 'Optimal Lag Range', 'Value': f"[{aggregate_df['Optimal Lag'].min():.0f}, {aggregate_df['Optimal Lag'].max():.0f}] days"}
])

display(Markdown("### 2. Variability Across 18 Stations"))
display(variability_summary.style.hide(axis='index').set_properties(**{'text-align': 'left'}))

In [ ]:
# 3. Modeling Recommendations
recommended_input_length = int(np.ceil(aggregate_df['Optimal Lag'].quantile(0.75)))
max_optimal_lag = int(aggregate_df['Optimal Lag'].max())
mean_corr_preliq = aggregate_df['Corr PRELIQ'].mean()
mean_corr_temp = aggregate_df['Corr T'].mean()
mean_corr_etp = aggregate_df['Corr ETP'].mean()
non_normal_count = sum(1 for r in normality_results if r['Normal Distribution'] == 'No')
seasonal_count = aggregate_df['Seasonal (365d)'].sum()
min_samples = aggregate_df['Samples'].min()

recommendations_md = f"""
### 3. Modeling Recommendations

#### A. Input Chunk Length
| Recommendation | Value | Rationale |
|----------------|-------|----------|
| **Primary** | **{recommended_input_length} days** | 75th percentile of optimal lags across all stations |
| Alternative | {max_optimal_lag} days | Maximum optimal lag (more conservative) |
| Rationale | — | Captures precipitation effects with sufficient lag |

#### B. Covariates
| Priority | Covariate | Mean Correlation | Recommendation |
|----------|-----------|------------------|----------------|
| **PRIMARY** | **PRELIQ_Q** | **{mean_corr_preliq:.3f}** | **Strongest predictor - MUST include** |
| Secondary | T_Q | {mean_corr_temp:.3f} | {'Moderate correlation - recommended' if abs(mean_corr_temp) > 0.3 else 'Weaker correlation - optional'} |
| Secondary | ETP_Q | {mean_corr_etp:.3f} | {'Moderate correlation - recommended' if abs(mean_corr_etp) > 0.3 else 'Weaker correlation - optional'} |

**Note:** Use lagged covariates (shift by optimal lag per station)

#### C. Data Normalization
| Aspect | Recommendation | Rationale |
|--------|----------------|----------|
| **Required** | **YES** | {non_normal_count}/4 variables in piezo1 are non-normal |
| Method | StandardScaler or MinMaxScaler | Both work well for time series |
| Strategy | Separate scalers per station | Accounts for station-specific ranges |

#### D. Trend Handling
| Metric | Value | Implication |
|--------|-------|-------------|
| Trend Variance | {100*var_trend/var_total:.1f}% | {'Trend is DOMINANT - models must handle trends well' if var_trend/var_total > 0.5 else 'Moderate trend - most models can handle'} |
| Recommended Models | RNN, LSTM, Transformer | Good at capturing long-term dependencies |
| Alternative | Detrending + simpler models | For linear/simple models only |

#### E. Seasonality
| Metric | Value | Recommendation |
|--------|-------|----------------|
| Seasonal Stations | {seasonal_count}/{len(aggregate_df)} ({100*seasonal_count/len(aggregate_df):.0f}%) | {'Strong seasonality - include time covariates' if seasonal_count > len(aggregate_df)/2 else 'Weak/no seasonality - time covariates optional'} |
| Time Features | month, day_of_year | {'RECOMMENDED' if seasonal_count > len(aggregate_df)/2 else 'Optional'} |

#### F. Model Architecture Suggestions
| Model Type | Suitability | Notes |
|------------|-------------|-------|
| **Transformers (PatchTST)** | **Excellent** | Best for long sequences and trends |
| **RNN/LSTM** | **Very Good** | Handle temporal dependencies well |
| Linear Models (DLinear) | Good | Fast baseline, may struggle with non-linearity |
| Ensemble | Excellent | Combine multiple approaches |

#### G. Train/Val/Test Split
| Aspect | Recommendation | Value |
|--------|----------------|---------|
| Split Ratio | Temporal split (no shuffling) | 70% train / 15% val / 15% test |
| Minimum Samples | Smallest station | {min_samples:,} samples |
| Strategy | Time-based | Preserve temporal order |

#### H. Station-Specific Considerations
| Issue | Count | Recommendation |
|-------|-------|----------------|
| Unusual lag patterns | {len(outlier_lag)} stations | Consider station-specific hyperparameters |
| Weak PRELIQ correlation | {len(low_corr_stations)} stations | May need different feature engineering |
| Non-stationary | {len(non_stationary)} stations | Consider differencing or detrending |
"""

display(Markdown(recommendations_md))

In [ ]:
# Save aggregate results to CSV
aggregate_df.to_csv(figs_dir / 'aggregate_analysis_results.csv', index=False)

# List all generated figures
display(Markdown("---"))
display(Markdown("## Generated Outputs"))

fig_files = sorted(figs_dir.glob('eda_*.png'))
fig_list = "\n".join([f"{i}. `{fig_file.name}`" for i, fig_file in enumerate(fig_files, 1)])

display(Markdown(f"""
### Figures ({len(fig_files)} total)
{fig_list}

### Data
1. `aggregate_analysis_results.csv`

**Location:** `{figs_dir}`
"""))

display(Markdown("---"))
display(Markdown("# Analysis Complete"))